# Cài đặt thuật toán ID3

*Bài 6, mục 6.1.3.* ID3 xây dựng cây từ trên xuống, đệ quy:
1. Nếu mọi mẫu cùng một lớp → nút lá với lớp đó.
2. Nếu không còn thuộc tính để chia → nút lá với lớp chiếm đa số.
3. Chọn thuộc tính có Information Gain lớn nhất, tạo một nhánh cho **mỗi giá trị** của thuộc tính, gọi đệ quy trên từng tập con (bỏ thuộc tính đã dùng).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path('..') / 'data'   # notebook chạy trong codes/
df = pd.read_csv(DATA / 'weather.csv')


def entropy(labels):
    """Entropy H(S) của một dãy nhãn (log cơ số 2)."""
    p = pd.Series(labels).value_counts(normalize=True).values   # tỉ lệ mỗi lớp p_i
    return 0.0 - np.sum(p * np.log2(p))     # 0.0 - ... để tránh in ra -0.000


def information_gain(data, attribute, target='play'):
    """IG(S, A) = H(S) - tổng có trọng số entropy của các tập con S_v."""
    weighted = sum(len(s) / len(data) * entropy(s[target]) for _, s in data.groupby(attribute))
    return entropy(data[target]) - weighted

Cây được biểu diễn bằng từ điển lồng nhau: `{thuộc_tính: {giá_trị: cây_con hoặc nhãn}}`.

In [2]:
def id3(data, attributes, target='play', depth=0):
    labels = data[target]
    if labels.nunique() == 1:                 # bước 1: nút tinh khiết
        return labels.iloc[0]
    if not attributes:                        # bước 2: hết thuộc tính
        return labels.mode()[0]
    gains = {a: information_gain(data, a, target) for a in attributes}
    best = max(gains, key=gains.get)          # bước 3: IG lớn nhất
    print('    ' * depth + 'Chọn %-11s (IG = %.3f) trên %d mẫu' % (best, gains[best], len(data)))
    remaining = [a for a in attributes if a != best]
    tree = {best: {}}
    for value, subset in data.groupby(best):
        tree[best][value] = id3(subset, remaining, target, depth + 1)
    return tree


attributes = ['weather', 'temperature', 'humidity', 'wind']
tree = id3(df, attributes)

Chọn weather     (IG = 0.400) trên 10 mẫu
    Chọn wind        (IG = 0.811) trên 4 mẫu
    Chọn temperature (IG = 0.918) trên 3 mẫu


Khi hai thuộc tính có IG bằng nhau (nhánh `Sunny`: `temperature` và `humidity` cùng bằng 0.918), `max` chọn thuộc tính xuất hiện trước trong danh sách. Ta đổi thứ tự để được cây giống giáo trình (chọn `humidity`).

In [3]:
tree = id3(df, ['weather', 'humidity', 'temperature', 'wind'])

Chọn weather     (IG = 0.400) trên 10 mẫu
    Chọn wind        (IG = 0.811) trên 4 mẫu
    Chọn humidity    (IG = 0.918) trên 3 mẫu


In [4]:
def print_tree(tree, indent=''):
    """In cây dạng thụt lề."""
    if not isinstance(tree, dict):
        print(indent + '-> ' + tree)
        return
    attribute = next(iter(tree))
    for value, subtree in tree[attribute].items():
        print(indent + '%s = %s' % (attribute, value))
        print_tree(subtree, indent + '    ')


print_tree(tree)

weather = Cloudy
    -> Yes
weather = Rainy
    wind = Strong
        -> No
    wind = Weak
        -> Yes
weather = Sunny
    humidity = High
        -> No
    humidity = Normal
        -> Yes


## Dự đoán bằng cây ID3

Đi từ gốc theo giá trị thuộc tính của mẫu cho tới khi gặp nút lá.

In [5]:
def predict(tree, sample):
    while isinstance(tree, dict):
        attribute = next(iter(tree))
        tree = tree[attribute].get(sample[attribute])
        if tree is None:          # giá trị chưa gặp khi huấn luyện
            return None
    return tree


df['predicted'] = [predict(tree, row) for _, row in df.iterrows()]
print('Độ chính xác trên tập huấn luyện:', (df['predicted'] == df['play']).mean())
new_day = {'weather': 'Rainy', 'temperature': 'Cool', 'humidity': 'High', 'wind': 'Weak'}
print('Ngày mới', new_day, '->', predict(tree, new_day))

Độ chính xác trên tập huấn luyện: 1.0
Ngày mới {'weather': 'Rainy', 'temperature': 'Cool', 'humidity': 'High', 'wind': 'Weak'} -> Yes
